In [0]:
try:
    dbutils.widgets.get("force_fail")
except Exception:
    dbutils.widgets.dropdown(
        "force_fail",
        "false",
        ["false", "true"],
        "Forçar falha"
    )

force_fail = dbutils.widgets.get("force_fail").lower() == "true"

expected_counts = {
    "coretec_dev.gold.dim_condominio": 36,
    "coretec_dev.gold.dim_data": 141,
    "coretec_dev.gold.fato_acessos_portaria": 6000,
    "coretec_dev.gold.fato_ocorrencias": 700
}

results = []
errors = []

for table_name, expected_count in expected_counts.items():
    actual_count = spark.table(table_name).count()
    status = "OK" if actual_count == expected_count else "FALHA"

    results.append(
        (table_name, expected_count, actual_count, status)
    )

    if actual_count != expected_count:
        errors.append(
            f"{table_name}: esperado={expected_count}, atual={actual_count}"
        )

display(
    spark.createDataFrame(
        results,
        ["tabela", "quantidade_esperada", "quantidade_atual", "status"]
    )
)

if force_fail:
    raise RuntimeError(
        "Falha controlada ativada para teste do Lakeflow Job."
    )

if errors:
    raise RuntimeError(
        "Validação do pipeline falhou: " + " | ".join(errors)
    )

print("VALIDAÇÃO FINAL: OK")